## 0. Configurando sessão spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-25247b04-6442-4984-9fb3-b63f746eaa44;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 243ms :: artifacts dl 6ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F, Window

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_gold_fato_meta = f"{par_source_project}.gold.fato_meta"
par_source_gold_fato_indicador_municipio = f"{par_source_project}.gold.fato_indicador_municipio"
par_source_gold_dim_municipio = f"{par_source_project}.gold.dim_municipio"
par_source_gold_dim_rede = f"{par_source_project}.gold.dim_rede"

par_source_gold_fato_resultados = f"{par_source_project}.gold.fato_resultados"

w = Window.partitionBy("nivel_geografico","local_id","rede_id","ano_meta").orderBy(F.col("ano_referencia").desc())

## 3. Leitura dos dados da origem

In [5]:
df_scr_meta = spark.read.format("bigquery").option("table",par_source_gold_fato_meta).load()
df_scr_indicador_municipio = spark.read.format("bigquery").option("table",par_source_gold_fato_indicador_municipio).load()
df_scr_municipio = spark.read.format("bigquery").option("table",par_source_gold_dim_municipio).load()
df_scr_rede = spark.read.format("bigquery").option("table",par_source_gold_dim_rede).load()

## 4. Transformações

In [6]:
meta_muni = (
    df_scr_meta
    .filter(F.col("nivel_geografico") == "municipio")
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
)

In [7]:
meta_ano = (
    meta_muni
    .select(
        F.col("local_id").alias("id_municipio"), "rede_id",
        F.col("ano_meta").alias("ano"),
        F.col("valor_meta").alias("meta_ano")
    )
)

In [8]:
meta_2030 = (
    meta_muni
    .filter(F.col("ano_meta") == 2030)
    .select(
        F.col("local_id").alias("id_municipio"), 
        "rede_id",
        F.col("valor_meta").alias("meta_2030")
    )
)

In [9]:
fato_resultados = (
    df_scr_indicador_municipio
    .join(df_scr_municipio,  "id_municipio", "left")
    .join(df_scr_rede, "rede_id",      "left")
    .join(meta_ano,  ["id_municipio","rede_id","ano"], "left")
    .join(meta_2030, ["id_municipio","rede_id"],       "left")
    .withColumn("gap_meta_ano",  F.round(F.col("taxa_alfabetizacao") - F.col("meta_ano"), 2))
    .withColumn("atingiu_meta_ano", F.col("taxa_alfabetizacao") >= F.col("meta_ano"))
    .withColumn("gap_ate_2030",   F.round(F.col("meta_2030") - F.col("taxa_alfabetizacao"), 2))
    .select(
        "ano","id_municipio","nome_municipio","sigla_uf","nome_uf","nome_regiao",
        "rede_id","rede",
        "taxa_alfabetizacao","media_portugues",
        "meta_ano","gap_meta_ano","atingiu_meta_ano",
        "meta_2030","gap_ate_2030"
    )
)

## 5. Armazenamento no BQ

In [10]:
(
    fato_resultados.write.format("bigquery")
    .option("table", par_source_gold_fato_resultados)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)

26/08/26 02:19:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                